# Carga de Telco Clean → PostgreSQL

## Esquema de tablas

```
dim_estado (id, descripcion)          → 'No', 'Sí', 'Sin servicio'
dim_genero (id, descripcion)          → 'Femenino', 'Masculino'
dim_contrato (id, descripcion)        → 'Month-to-month', 'One year', 'Two year'
dim_metodo_pago (id, descripcion)     → 4 métodos de pago
dim_internet (id, descripcion)        → 'No', 'DSL', 'Fiber optic'
cliente (PK + FKs + campos numéricos)
```

## Requisitos
```
pip install pandas sqlalchemy psycopg2-binary python-dotenv
```

## Archivo .env requerido
Crea un archivo `.env` en la misma carpeta con:
```
DB_HOST=localhost
DB_PORT=5432
DB_NAME=telco_db
DB_USER=postgres
DB_PASSWORD=
```

In [1]:
%pip install pandas sqlalchemy psycopg2-binary python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Importar librerías

In [2]:
import pandas as pd
import os
import logging
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


## 2. Configurar logger

In [3]:
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/carga_telco.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Inicio del proceso de carga Telco → PostgreSQL")
print("Logger configurado")

Logger configurado


## 3. Conectar a PostgreSQL

In [4]:
from urllib.parse import quote_plus
load_dotenv(".env")

DB_HOST     = os.getenv("DB_HOST",     "localhost")
DB_PORT     = os.getenv("DB_PORT",     "5432")
DB_NAME     = os.getenv("DB_NAME",     "telco_db")
DB_USER     = os.getenv("DB_USER",     "postgres")
DB_PASSWORD = quote_plus(os.getenv("DB_PASSWORD", ""))

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT current_database(), current_schema();"))
        db, schema = result.fetchone()
        logging.info(f"Conexión exitosa → DB: {db}, Schema: {schema}")
        print(f"Conectado a: base={db}, schema={schema}")
except Exception as e:
    logging.error(f"Error de conexión: {e}")
    print(f"Error de conexión: {e}")
    raise

Conectado a: base=telco_db, schema=public


## 4. Crear tablas dimensionales y tabla cliente

In [5]:
sql_crear_tablas = """
-- Tabla de estados: No (0), Sí (1), Sin servicio (2)
CREATE TABLE IF NOT EXISTS dim_estado (
    id          SMALLINT PRIMARY KEY,
    descripcion VARCHAR(20) NOT NULL
);

-- Tabla de género: Femenino (0), Masculino (1)
CREATE TABLE IF NOT EXISTS dim_genero (
    id          SMALLINT PRIMARY KEY,
    descripcion VARCHAR(20) NOT NULL
);

-- Tabla de tipo de contrato
CREATE TABLE IF NOT EXISTS dim_contrato (
    id          SMALLINT PRIMARY KEY,
    descripcion VARCHAR(30) NOT NULL
);

-- Tabla de método de pago
CREATE TABLE IF NOT EXISTS dim_metodo_pago (
    id          SMALLINT PRIMARY KEY,
    descripcion VARCHAR(40) NOT NULL
);

-- Tabla de tipo de servicio de internet
CREATE TABLE IF NOT EXISTS dim_internet (
    id          SMALLINT PRIMARY KEY,
    descripcion VARCHAR(20) NOT NULL
);

-- Tabla principal de clientes
CREATE TABLE IF NOT EXISTS cliente (
    customer_id         VARCHAR(20)  PRIMARY KEY,
    genero_id           SMALLINT     REFERENCES dim_genero(id),
    senior_citizen      BOOLEAN      NOT NULL,
    partner             BOOLEAN      NOT NULL,
    dependents          BOOLEAN      NOT NULL,
    tenure              SMALLINT     NOT NULL,
    phone_service       BOOLEAN      NOT NULL,
    multiple_lines_id   SMALLINT     REFERENCES dim_estado(id),
    internet_id         SMALLINT     REFERENCES dim_internet(id),
    online_security_id  SMALLINT     REFERENCES dim_estado(id),
    online_backup_id    SMALLINT     REFERENCES dim_estado(id),
    device_protection_id SMALLINT    REFERENCES dim_estado(id),
    tech_support_id     SMALLINT     REFERENCES dim_estado(id),
    streaming_tv_id     SMALLINT     REFERENCES dim_estado(id),
    streaming_movies_id SMALLINT     REFERENCES dim_estado(id),
    contrato_id         SMALLINT     REFERENCES dim_contrato(id),
    paperless_billing   BOOLEAN      NOT NULL,
    metodo_pago_id      SMALLINT     REFERENCES dim_metodo_pago(id),
    monthly_charges     NUMERIC(8,2) NOT NULL,
    total_charges       NUMERIC(10,2),
    churn               BOOLEAN      NOT NULL
);
"""

try:
    with engine.begin() as conn:
        conn.execute(text(sql_crear_tablas))
    logging.info("Tablas creadas/verificadas correctamente")
    print("Tablas creadas o verificadas correctamente")
except Exception as e:
    logging.error(f"Error creando tablas: {e}")
    print(f"Error: {e}")
    raise

Tablas creadas o verificadas correctamente


## 5. Insertar datos en tablas dimensionales

In [6]:
# Definir catálogos según el encoding del CSV limpio
catalogos = {
    "dim_estado": [
        {"id": 0, "descripcion": "No"},
        {"id": 1, "descripcion": "Sí"},
        {"id": 2, "descripcion": "Sin servicio"}
    ],
    "dim_genero": [
        {"id": 0, "descripcion": "Femenino"},
        {"id": 1, "descripcion": "Masculino"}
    ],
    "dim_contrato": [
        {"id": 0, "descripcion": "Month-to-month"},
        {"id": 1, "descripcion": "One year"},
        {"id": 2, "descripcion": "Two year"}
    ],
    "dim_metodo_pago": [
        {"id": 0, "descripcion": "Bank transfer (automatic)"},
        {"id": 1, "descripcion": "Credit card (automatic)"},
        {"id": 2, "descripcion": "Electronic check"},
        {"id": 3, "descripcion": "Mailed check"}
    ],
    "dim_internet": [
        {"id": 0, "descripcion": "No"},
        {"id": 1, "descripcion": "DSL"},
        {"id": 2, "descripcion": "Fiber optic"}
    ]
}

try:
    with engine.begin() as conn:
        for tabla, filas in catalogos.items():
            df_dim = pd.DataFrame(filas)
            # INSERT ... ON CONFLICT DO NOTHING para no duplicar al re-ejecutar
            for _, row in df_dim.iterrows():
                conn.execute(
                    text(f"INSERT INTO {tabla} (id, descripcion) VALUES (:id, :descripcion) ON CONFLICT DO NOTHING"),
                    {"id": row["id"], "descripcion": row["descripcion"]}
                )
            print(f"  {tabla}: {len(filas)} registros insertados/verificados")
    logging.info("Catálogos dimensionales cargados")
    print("\nCatálogos cargados correctamente")
except Exception as e:
    logging.error(f"Error cargando catálogos: {e}")
    print(f"Error: {e}")
    raise

  dim_estado: 3 registros insertados/verificados
  dim_genero: 2 registros insertados/verificados
  dim_contrato: 3 registros insertados/verificados
  dim_metodo_pago: 4 registros insertados/verificados
  dim_internet: 3 registros insertados/verificados

Catálogos cargados correctamente


## 6. Leer y preparar el CSV limpio

In [ ]:
# Ajusta la ruta si tu CSV está en otra ubicación

df = pd.read_csv('../data/processed/Teleco_validado.csv')
print(f"Filas leídas: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
df.head(3)

Filas leídas: 7043
Columnas: ['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn']


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,0,0,1,0,1,0,2,1,0,...,0,0,0,0,0,1,0,29.85,29.85,0
1,5575-GNVDE,1,0,0,0,34,1,0,1,1,...,1,0,0,0,1,0,1,56.95,1889.50,0
2,3668-QPYBK,1,0,0,0,2,1,0,1,1,...,0,0,0,0,0,1,1,53.85,108.15,1


In [9]:
# Renombrar columnas al esquema de la tabla cliente
df_cliente = df.rename(columns={
    "gender":              "genero_id",
    "multiple_lines":      "multiple_lines_id",
    "internet_service":    "internet_id",
    "online_security":     "online_security_id",
    "online_backup":       "online_backup_id",
    "device_protection":   "device_protection_id",
    "tech_support":        "tech_support_id",
    "streaming_tv":        "streaming_tv_id",
    "streaming_movies":    "streaming_movies_id",
    "contract":            "contrato_id",
    "payment_method":      "metodo_pago_id",
})

# Convertir columnas booleanas (0/1 → True/False)
bool_cols = ["senior_citizen", "partner", "dependents",
             "phone_service", "paperless_billing", "churn"]
for col in bool_cols:
    df_cliente[col] = df_cliente[col].astype(bool)

print("Columnas finales del DataFrame:")
print(df_cliente.dtypes)
df_cliente.head(3)

Columnas finales del DataFrame:
customer_id                 str
genero_id                 int64
senior_citizen             bool
partner                    bool
dependents                 bool
tenure                    int64
phone_service              bool
multiple_lines_id         int64
internet_id               int64
online_security_id        int64
online_backup_id          int64
device_protection_id      int64
tech_support_id           int64
streaming_tv_id           int64
streaming_movies_id       int64
contrato_id               int64
paperless_billing          bool
metodo_pago_id            int64
monthly_charges         float64
total_charges           float64
churn                      bool
dtype: object


,customer_id,genero_id,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines_id,internet_id,online_security_id,...,device_protection_id,tech_support_id,streaming_tv_id,streaming_movies_id,contrato_id,paperless_billing,metodo_pago_id,monthly_charges,total_charges,churn
0,7590-VHVEG,0,False,True,False,1,False,2,1,0,...,0,0,0,0,0,True,0,29.85,29.85,False
1,5575-GNVDE,1,False,False,False,34,True,0,1,1,...,1,0,0,0,1,False,1,56.95,1889.50,False
2,3668-QPYBK,1,False,False,False,2,True,0,1,1,...,0,0,0,0,0,True,1,53.85,108.15,True


## 7. Cargar tabla cliente

In [ ]:
try:
    filas_insertadas = df_cliente.to_sql(
        name="cliente",
        con=engine,
        if_exists="append",   # 'replace' borra y recrea; 'append' agrega
        index=False,
        method="multi",       # inserciones en lote (más rápido)
        chunksize=500
    )
    logging.info(f"Tabla cliente cargada: {filas_insertadas} filas")
    print(f"Tabla cliente cargada: {filas_insertadas} filas insertadas")
except Exception as e:
    logging.error(f"Error cargando tabla cliente: {e}")
    print(f"Error: {e}")
    raise

## 8. Verificar carga

In [11]:
queries_verificacion = {
    "Total clientes":        "SELECT COUNT(*) FROM cliente;",
    "Clientes con churn":    "SELECT COUNT(*) FROM cliente WHERE churn = TRUE;",
    "Tipos de contrato":     """
        SELECT c.descripcion, COUNT(*) AS total
        FROM cliente cl
        JOIN dim_contrato c ON cl.contrato_id = c.id
        GROUP BY c.descripcion ORDER BY total DESC;
    """,
    "Servicio de internet":  """
        SELECT i.descripcion, COUNT(*) AS total
        FROM cliente cl
        JOIN dim_internet i ON cl.internet_id = i.id
        GROUP BY i.descripcion ORDER BY total DESC;
    """
}

with engine.connect() as conn:
    for nombre, sql in queries_verificacion.items():
        result = pd.read_sql(text(sql), conn)
        print(f"\n=== {nombre} ===")
        print(result.to_string(index=False))

logging.info("Verificación completada")
print("\nCarga completada exitosamente")


=== Total clientes ===
 count
  7043

=== Clientes con churn ===
 count
  1869

=== Tipos de contrato ===
   descripcion  total
Month-to-month   3875
      Two year   1695
      One year   1473

=== Servicio de internet ===
descripcion  total
Fiber optic   3096
        DSL   2421
         No   1526

Carga completada exitosamente
